<a href="https://colab.research.google.com/github/BuaKot/67010005.github.io/blob/main/Coral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# -*- coding: utf-8 -*-
"""
Coral bleaching early-warning pipeline (v2, cleaned)
====================================================
ทำนาย "DHW >= 4 ในอีก H วัน" สำหรับ Koh Man Nai / Koh Phi Phi
ประเมินแบบ leave-one-island-out (เทรนเกาะหนึ่ง ทดสอบอีกเกาะ)

ต่างจากเวอร์ชันเดิม:
  1. ไม่ fit scaler บนข้อมูลทั้งสองเกาะ (ไม่มี leakage), scaler fit จาก train เท่านั้น
  2. ใช้ feature หน่วยฟิสิกส์ (hotspot = SST - MMM, DHW ประมาณจาก SST) เทียบข้ามเกาะได้
  3. มี validation แบบแบ่งตามเวลา + EarlyStopping + (ถ้าเป็นไปได้) Platt calibration
  4. เลือก threshold จาก validation ด้วย F2 (ไม่ใช้ probability บน train)
  5. metric หลักคือ "onset-only" (เฉพาะวันที่ DHW<4 ตอนนี้) เทียบกับ baseline จริง:
     persistence และกฎ "DHW>=3 และกำลังเพิ่ม"
  6. รันหลาย seed รายงาน mean ± sd
  7. event report นับ lead time เฉพาะ alert ในหน้าต่าง 30 วันก่อนเหตุการณ์
  8. จัดการข้อมูลขาดวัน (window ไม่ข้ามช่องว่าง) และบันทึกโมเดล/scaler/ผลลัพธ์

ข้อจำกัดที่ต้องเขียนในรายงาน:
  * label = (DHW >= 4) ไม่ใช่ข้อมูลฟอกขาวจริงจากการสำรวจ
  * ถ้าแต่ละเกาะมี event เดียว ผลทุกตัวเลขมี variance สูงมาก
  * MMM ที่ใช้เป็นค่าปรับให้พอดีกับ DHW ในข้อมูล ไม่ใช่ค่าทางการของ NOAA

วิธีใช้ (Colab):  !pip install tensorflow scikit-learn pandas numpy joblib  แล้วรันไฟล์นี้
"""
import json
import os
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             roc_auc_score)
from sklearn.preprocessing import StandardScaler

# ======================================================================
# CONFIG
# ======================================================================
FILES = {
    "man_nai": "final_data_KohManNai_1.csv",
    "phi_phi": "final_data_KohPhiPhi.csv",
}
# MMM ที่ปรับให้พอดีกับ DHW ในข้อมูล (ควรเทียบกับ NOAA CRW climatology)
MMM = {"man_nai": 29.70, "phi_phi": 30.40}

H = 7                  # ทำนายล่วงหน้ากี่วัน
T = 30                 # ความยาว window (ลอง 60/84 ได้)
DHW_THRESHOLD = 4.0
DHW_WINDOW = 84        # DHW สะสม 12 สัปดาห์ = 84 วัน
BAND_COLS = ["blue", "green", "red", "nir"]
USE_BANDS = True       # ตั้ง False เพื่อทำ ablation (SST อย่างเดียว)
USE_SEASON = False     # sin/cos วันในปี: ถ้ามี event เดียวต่อเกาะ โมเดลจะจำฤดูกาลแทนฟิสิกส์ จึงปิดไว้

N_SEEDS = 5
EPOCHS = 60
BATCH = 32
PATIENCE = 8
VAL_FRAC = 0.2         # ส่วนท้ายของ train (ตามเวลา) ใช้เป็น validation
MAX_POS_W = 10.0       # เพดาน class weight
MIN_VAL_POS = 10       # ต้องมี positive/negative ใน val อย่างน้อยเท่านี้ ถึงจะ calibrate/เลือก threshold
EVENT_WINDOW = 30      # นับ alert ก่อนเหตุการณ์ไม่เกินกี่วันว่า "เตือนล่วงหน้าที่ถูกต้อง"
RULE_DHW = 3.0         # กฎ baseline: DHW >= 3 และกำลังเพิ่มขึ้น

OUT_DIR = "results"
EXPORT_ISLAND_STATE = True


# ======================================================================
# 1. DATA
# ======================================================================
def feature_columns():
    cols = ["hotspot", "anom", "dhw_hat", "dhw_hat_d7"]
    if USE_SEASON:
        cols += ["doy_sin", "doy_cos"]
    if USE_BANDS:
        cols += BAND_COLS
    return cols


def load_island(path, mmm):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").drop_duplicates("date").reset_index(drop=True)

    if "bleaching_label" not in df.columns:
        df["bleaching_label"] = (df["dhw"] >= DHW_THRESHOLD).astype(int)

    need = ["sst", "anom", "dhw"] + (BAND_COLS if USE_BANDS else [])
    df[need] = df[need].interpolate(limit=3, limit_area="inside")  # อุดช่องว่างสั้น ๆ (เช่น เมฆ)
    df = df.dropna(subset=need + ["bleaching_label"]).reset_index(drop=True)

    # --- feature ที่คำนวณจากอดีตเท่านั้น (ไม่ใช้ dhw จริงเป็น input) ---
    hs = df["sst"] - mmm
    df["hotspot"] = hs
    df["dhw_hat"] = hs.where(hs >= 1, 0).rolling(DHW_WINDOW).sum() / 7.0   # NaN ช่วง warm-up
    df["dhw_hat_d7"] = df["dhw_hat"].diff(7)
    doy = df["date"].dt.dayofyear
    df["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    df = df.dropna(subset=["dhw_hat", "dhw_hat_d7"]).reset_index(drop=True)

    # segment = ช่วงที่วันที่ต่อเนื่องกัน (window จะไม่ข้ามช่องว่าง)
    df["segment"] = (df["date"].diff().dt.days != 1).cumsum()
    n_seg = df["segment"].nunique()
    if n_seg > 1:
        warnings.warn(f"{path}: พบข้อมูลขาดวัน แบ่งเป็น {n_seg} ช่วงต่อเนื่อง")
    return df


def make_windows(df, cols, t=T, h=H):
    """สร้าง (X, y, meta) โดย window ไม่ข้ามช่องว่างของวันที่"""
    assert t >= 8, "T ต้อง >= 8 เพราะใช้ DHW ย้อนหลัง 7 วันใน baseline"
    F = df[cols].values.astype("float32")
    lab = df["bleaching_label"].values.astype(int)
    dhw = df["dhw"].values
    anom = df["anom"].values
    X, y, d_now, d_prev, a_now = [], [], [], [], []
    for pos in df.groupby("segment").indices.values():
        s, n = pos[0], len(pos)
        for i in range(n - t - h + 1):
            a = s + i
            last = a + t - 1
            X.append(F[a:a + t])
            y.append(lab[last + h])
            d_now.append(dhw[last])
            d_prev.append(dhw[last - 7])
            a_now.append(anom[last])
    meta = dict(dhw_now=np.array(d_now), dhw_prev=np.array(d_prev), anom_now=np.array(a_now))
    return np.array(X, dtype="float32"), np.array(y, dtype=int), meta


def scale_windows(X, scaler):
    n, t, f = X.shape
    return scaler.transform(X.reshape(-1, f)).reshape(n, t, f).astype("float32")


# ======================================================================
# 2. MODEL
# ======================================================================
def build_model(t, n_feat):
    from tensorflow.keras import Model, layers, optimizers, metrics
    inp = layers.Input(shape=(t, n_feat))
    x = layers.Conv1D(32, 3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    # ใช้ state สุดท้ายของ BiLSTM แทน average pooling เพื่อไม่ให้วันล่าสุดถูกเฉลี่ยทิ้ง
    x = layers.Bidirectional(layers.LSTM(32))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = Model(inp, out)
    m.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=[metrics.AUC(name="auc")])
    return m


def fit_predict(Xtr, ytr, Xva, yva, Xte, seed, use_val):
    """เทรน 1 seed -> คืน (model, p_val, p_test)"""
    import tensorflow as tf
    from tensorflow.keras import callbacks
    tf.keras.utils.set_random_seed(seed)

    pos = ytr.sum()
    pos_w = min((len(ytr) - pos) / pos, MAX_POS_W) if pos > 0 else 1.0
    m = build_model(Xtr.shape[1], Xtr.shape[2])
    es = callbacks.EarlyStopping(monitor="val_loss" if use_val else "loss",
                                 patience=PATIENCE, restore_best_weights=True)
    kw = dict(validation_data=(Xva, yva)) if use_val else {}
    m.fit(Xtr, ytr, epochs=EPOCHS, batch_size=BATCH, verbose=0,
          class_weight={0: 1.0, 1: float(pos_w)}, callbacks=[es], **kw)
    pv = m.predict(Xva, verbose=0).ravel() if len(Xva) else np.array([])
    pt = m.predict(Xte, verbose=0).ravel()
    return m, pv, pt


# ======================================================================
# 3. METRICS
# ======================================================================
def safe_auc(y, s):
    return roc_auc_score(y, s) if 0 < y.sum() < len(y) else float("nan")


def safe_pr(y, s):
    return average_precision_score(y, s) if y.sum() > 0 else float("nan")


def rec_prec(y, pred):
    tp = int(((pred == 1) & (y == 1)).sum())
    rec = tp / y.sum() if y.sum() else float("nan")
    prec = tp / pred.sum() if pred.sum() else 0.0
    return rec, prec


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def pick_threshold_f2(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    prec, rec = prec[:-1], rec[:-1]
    f2 = 5 * prec * rec / np.maximum(4 * prec + rec, 1e-12)
    return float(thr[np.argmax(f2)])


def find_events(y):
    """คืนรายการ (start, end) ของช่วงที่ y==1 ต่อเนื่องกัน"""
    ev, i, n = [], 0, len(y)
    while i < n:
        if y[i] == 1:
            j = i
            while j + 1 < n and y[j + 1] == 1:
                j += 1
            ev.append((i, j))
            i = j + 1
        else:
            i += 1
    return ev


def event_report(y, pred, window=EVENT_WINDOW):
    """lead time นับจาก alert แรกที่อยู่ในหน้าต่าง [start-window, start] เท่านั้น"""
    events = find_events(y)
    idx = np.where(pred == 1)[0]
    rows, zone = [], np.zeros(len(y), dtype=bool)
    for s, e in events:
        zone[max(0, s - window): e + 1] = True
        pre = idx[(idx >= s - window) & (idx <= s)]
        inside = idx[(idx >= s) & (idx <= e)]
        if len(pre):
            status, val = "early", int(s - pre[0])
        elif len(inside):
            status, val = "late", int(inside[0] - s)
        else:
            status, val = "missed", None
        rows.append(dict(start=int(s), end=int(e), status=status, days=val,
                         coverage=float(len(inside) / (e - s + 1))))
    false_alarm = int((~zone[idx]).sum()) if len(idx) else 0
    return rows, false_alarm


# ======================================================================
# 4. ONE FOLD
# ======================================================================
def run_fold(train_df, test_df, name, fit_fn=fit_predict):
    cols = feature_columns()
    cut = int(len(train_df) * (1 - VAL_FRAC))
    tr_part = train_df.iloc[:cut].reset_index(drop=True)
    va_part = train_df.iloc[cut:].reset_index(drop=True)   # แยกตามเวลา, window ไม่ซ้อนกับ train
    Xtr, ytr, _ = make_windows(tr_part, cols)
    if ytr.sum() < MIN_VAL_POS:
        # ถ้าแบ่ง val ออกแล้ว train เหลือ positive น้อยเกินไป (เช่น event อยู่ท้ายข้อมูล) ให้ไม่แบ่ง val
        print("⚠️ แบ่ง validation แล้ว train เหลือ positive น้อยเกินไป: ใช้ train ทั้งหมด ไม่มี validation")
        tr_part = train_df.reset_index(drop=True)
        Xtr, ytr, _ = make_windows(tr_part, cols)
        Xva, yva = np.zeros((0, T, len(cols)), dtype="float32"), np.zeros(0, dtype=int)
    else:
        Xva, yva, _ = make_windows(va_part, cols) if len(va_part) > T + H else (
            np.zeros((0, T, len(cols)), dtype="float32"), np.zeros(0, dtype=int), None)

    scaler = StandardScaler().fit(tr_part[cols].values)   # fit จาก train เท่านั้น
    Xte, yte, meta = make_windows(test_df, cols)
    Xtr, Xva, Xte = (scale_windows(a, scaler) if len(a) else a for a in (Xtr, Xva, Xte))

    val_ok = (len(yva) > 0 and yva.sum() >= MIN_VAL_POS and (len(yva) - yva.sum()) >= MIN_VAL_POS)
    print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
    print(f"Train {len(ytr)} windows (+ {int(ytr.sum())}) | Val {len(yva)} (+ {int(yva.sum())}) "
          f"| Test {len(yte)} (+ {int(yte.sum())})")
    if not val_ok:
        print("⚠️ validation มี positive/negative น้อยเกินไป: ใช้ threshold 0.5, ไม่ calibrate, "
              "EarlyStopping ดูจาก train loss")
    if ytr.sum() == 0 or yte.sum() == 0:
        print("⚠️ ไม่มี positive ใน train หรือ test: metric หลายตัวจะเป็น NaN")

    # --- เทรนหลาย seed ---
    pv_all, pt_all, seed_metrics = [], [], []
    onset = meta["dhw_now"] < DHW_THRESHOLD     # วันที่ยังไม่ถึงเกณฑ์ตอนนี้
    yo = yte[onset]
    for seed in range(N_SEEDS):
        m, pv, pt = fit_fn(Xtr, ytr, Xva, yva, Xte, 42 + seed, val_ok)
        pv_all.append(pv)
        pt_all.append(pt)
        seed_metrics.append(dict(auc=safe_auc(yte, pt), pr=safe_pr(yte, pt),
                                 auc_on=safe_auc(yo, pt[onset]), pr_on=safe_pr(yo, pt[onset])))
        if hasattr(m, "save") and seed == 0:
            os.makedirs(OUT_DIR, exist_ok=True)
            m.save(os.path.join(OUT_DIR, f"model_{name.split()[0]}.keras"))
    joblib.dump(scaler, os.path.join(OUT_DIR, f"scaler_{name.split()[0]}.joblib"))

    pv, pt = np.mean(pv_all, axis=0), np.mean(pt_all, axis=0)

    # --- calibration + threshold จาก validation ---
    thr = 0.5
    if val_ok:
        cal = LogisticRegression(C=1e6).fit(logit(pv).reshape(-1, 1), yva)
        pv = cal.predict_proba(logit(pv).reshape(-1, 1))[:, 1]
        pt = cal.predict_proba(logit(pt).reshape(-1, 1))[:, 1]
        thr = pick_threshold_f2(yva, pv)

    # --- รายงาน ---
    ms = pd.DataFrame(seed_metrics)
    print("\n[ความสามารถในการจัดอันดับ: mean ± sd จาก %d seed]" % N_SEEDS)
    print(f"  ทุกวัน      : AUC {ms.auc.mean():.3f}±{ms.auc.std():.3f} | PR-AUC {ms.pr.mean():.3f}±{ms.pr.std():.3f}")
    print(f"  onset-only  : AUC {ms.auc_on.mean():.3f}±{ms.auc_on.std():.3f} | PR-AUC {ms.pr_on.mean():.3f}±{ms.pr_on.std():.3f}"
          f"   <-- ตัวเลขหลัก (วันที่ DHW<{DHW_THRESHOLD:g}: {int(onset.sum())} วัน, "
          f"ข้ามเกณฑ์ใน {H} วัน: {int(yo.sum())} วัน)")

    base = {
        "persistence (DHW วันนี้)": meta["dhw_now"],
        "rule (DHW วันนี้ ถ้ากำลังเพิ่ม)": np.where(meta["dhw_now"] > meta["dhw_prev"], meta["dhw_now"], 0.0),
        "SST anomaly วันนี้": meta["anom_now"],
    }
    print("\n[Baselines บน onset-only]")
    for k, s in base.items():
        print(f"  {k:<32}: AUC {safe_auc(yo, s[onset]):.3f} | PR-AUC {safe_pr(yo, s[onset]):.3f}")

    pred = (pt >= thr).astype(int)
    rule_pred = ((meta["dhw_now"] >= RULE_DHW) & (meta["dhw_now"] > meta["dhw_prev"])).astype(int)
    print(f"\n[ที่ threshold โมเดล = {thr:.3f}{' (F2 จาก val)' if val_ok else ' (default)'}]")
    for lbl, p_ in [("โมเดล", pred), (f"กฎ DHW>={RULE_DHW:g}&เพิ่ม", rule_pred)]:
        r_all, p_all = rec_prec(yte, p_)
        r_on, p_on = rec_prec(yo, p_[onset])
        print(f"  {lbl:<18}: ทุกวัน recall {r_all:.2f}/prec {p_all:.2f} | "
              f"onset recall {r_on:.2f}/prec {p_on:.2f} | วันเตือน {int(p_.sum())}")

    print("\n[Event report (โมเดล)]")
    summary_events = {}
    for lbl, p_ in [("โมเดล", pred), ("กฎ", rule_pred)]:
        rows, fa = event_report(yte, p_)
        summary_events[lbl] = dict(events=rows, false_alarm_days=fa)
        for r in rows:
            txt = {"early": f"เตือนก่อน {r['days']} วัน", "late": f"เตือนช้า {r['days']} วัน",
                   "missed": "ไม่เตือน"}[r["status"]]
            print(f"  {lbl:<6} event [{r['start']}-{r['end']}]: {txt} | ครอบคลุม {r['coverage']:.0%} "
                  f"| false alarm (นอก ±{EVENT_WINDOW} วัน): {fa} วัน")

    return dict(name=name, threshold=thr, val_ok=bool(val_ok),
                seed_metrics=seed_metrics, events=summary_events)


# ======================================================================
# 5. EXPORT STATE FOR API
# ======================================================================
def export_island_state(path="island_state.json"):
    state = {}
    for key, csv in FILES.items():
        df = pd.read_csv(csv)
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)
        state[key] = {
            "mmm": MMM[key],
            "sst_last84": [round(float(v), 3) for v in df["sst"].values[-84:]],
            "dhw_now": float(df["dhw"].iloc[-1]),
            "dhw_7d_ago": float(df["dhw"].iloc[-8]),
            "date_last": str(df["date"].iloc[-1].date()),
        }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=1)
    print(f"\nบันทึก {path}")


# ======================================================================
# MAIN
# ======================================================================
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    mn = load_island(FILES["man_nai"], MMM["man_nai"])
    pp = load_island(FILES["phi_phi"], MMM["phi_phi"])
    print(f"Man Nai {mn['date'].min().date()} → {mn['date'].max().date()} ({len(mn)} วัน)")
    print(f"Phi Phi {pp['date'].min().date()} → {pp['date'].max().date()} ({len(pp)} วัน)")

    results = [
        run_fold(mn, pp, "FoldA Train=ManNai Test=PhiPhi"),
        run_fold(pp, mn, "FoldB Train=PhiPhi Test=ManNai"),
    ]
    with open(os.path.join(OUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=1, default=float)
    if EXPORT_ISLAND_STATE:
        export_island_state(os.path.join(OUT_DIR, "island_state.json"))


if __name__ == "__main__":
    main()

/tmp/ipykernel_1355/1438404716.py:108: UserWarning: final_data_KohPhiPhi.csv: พบข้อมูลขาดวัน แบ่งเป็น 2 ช่วงต่อเนื่อง
  warnings.warn(f"{path}: พบข้อมูลขาดวัน แบ่งเป็น {n_seg} ช่วงต่อเนื่อง")


Man Nai 2023-04-01 → 2025-12-31 (1006 วัน)
Phi Phi 2022-04-01 → 2026-09-14 (1627 วัน)

FoldA Train=ManNai Test=PhiPhi
Train 768 windows (+ 71) | Val 166 (+ 0) | Test 1555 (+ 79)
⚠️ validation มี positive/negative น้อยเกินไป: ใช้ threshold 0.5, ไม่ calibrate, EarlyStopping ดูจาก train loss

[ความสามารถในการจัดอันดับ: mean ± sd จาก 5 seed]
  ทุกวัน      : AUC 0.946±0.056 | PR-AUC 0.724±0.232
  onset-only  : AUC 0.982±0.035 | PR-AUC 0.604±0.395   <-- ตัวเลขหลัก (วันที่ DHW<4: 1476 วัน, ข้ามเกณฑ์ใน 7 วัน: 7 วัน)

[Baselines บน onset-only]
  persistence (DHW วันนี้)        : AUC 0.997 | PR-AUC 0.555
  rule (DHW วันนี้ ถ้ากำลังเพิ่ม) : AUC 1.000 | PR-AUC 1.000
  SST anomaly วันนี้              : AUC 0.998 | PR-AUC 0.510

[ที่ threshold โมเดล = 0.500 (default)]
  โมเดล             : ทุกวัน recall 1.00/prec 0.65 | onset recall 1.00/prec 0.14 | วันเตือน 122
  กฎ DHW>=3&เพิ่ม   : ทุกวัน recall 0.68/prec 0.92 | onset recall 1.00/prec 0.58 | วันเตือน 59

[Event report (โมเดล)]
  โมเดล  event [731-

In [ ]:
!pip install requests pandas numpy
%run fetch_crw_data.py


== phi_phi (7.74, 98.77) ==
  pixel ที่ใช้: (7.725, 98.775) ห่างจากจุดที่ระบุ ~1.8 กม.
  phi_phi 2010-2014: OK (79s)
    ลองใหม่ (coastwatch ครั้งที่ 1): HTTP 408: Error {
    code=408;
    message="Request Timeout: TimeoutException: Timeout waiting for your other requests to process. Please make just one request
    ลองใหม่ (coastwatch ครั้งที่ 1): ReadTimeout หลัง 242s
    ลองใหม่ (coastwatch ครั้งที่ 1): ReadTimeout หลัง 242s
    ลองใหม่ (coastwatch ครั้งที่ 2): HTTP 408: Error {
    code=408;
    message="Request Timeout: TimeoutException: Timeout waiting for your other requests to process. Please make just one request
    ลองใหม่ (coastwatch ครั้งที่ 2): HTTP 408: Error {
    code=408;
    message="Request Timeout: TimeoutException: Timeout waiting for your other requests to process. Please make just one request
    ลองใหม่ (coastwatch ครั้งที่ 2): HTTP 408: Error {
    code=408;
    message="Request Timeout: TimeoutException: Timeout waiting for your other requests to process. P